# Pancreas

We evaluate FFM on pancreatic endocrinogenesis (Bastidas-Ponce et al., 2019). For pseudotime hold-out we withhold the middle tertile following Petrović et al. (2025); the RNA velocity, transition graph and derived geometric quantities are recomputed after removing the evaluation cells. FFM uses a velocity-derived transition kernel, and hyperparameters are selected independently for each representation using a validation subset disjoint from the reported withheld cells.

Reproduces the Pancreas entries of Tables 1, 2 and 10, Figure 4, and Tables 4, 5, 6 and 8.

In [1]:
import pathlib
import sys

ROOT = next(p for p in (pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents)
            if (p / "ffm").is_dir() and (p / "scripts").is_dir())
sys.path.insert(0, str(ROOT))

from scripts.pancreas import paper

paper.ensure()

## Midpoint and endpoint $W_2$

Euclidean $W_2$ at $t = 1/2$ against the 1,112 withheld cells not used for tuning and at $t = 1$ against the target; mean ± population SD over five seeds at the tuned points. Table 1 (top), Table 4 and Figure 4(a) come from the same run set. The sampling floor is the empirical Wasserstein distance between two disjoint halves of the corresponding reference population.

In [3]:
paper.mid_end()

mid W2         end W2
d  Method                                               
2  OT-CFM                   1.762 ± 0.022  0.162 ± 0.018
   MFM / LAND               1.481 ± 0.017  0.325 ± 0.103
   Curly-FM                 1.690 ± 0.043  0.131 ± 0.010
   FFM (ours)               1.695 ± 0.152  0.306 ± 0.056
   FFM (ours), with noise   1.749 ± 0.135  0.272 ± 0.038
   sampling floor                   0.111          0.102
20 OT-CFM                  12.335 ± 0.052  5.156 ± 0.208
   MFM / LAND              11.908 ± 0.297  8.345 ± 0.847
   Curly-FM                12.370 ± 0.076  4.952 ± 0.200
   FFM (ours)              10.978 ± 0.166  5.263 ± 0.231
   FFM (ours), with noise  10.942 ± 0.075  5.120 ± 0.438
   sampling floor                   4.178          4.448
50 OT-CFM                  13.496 ± 0.107  8.007 ± 0.173
   MFM / LAND              13.104 ± 0.127  8.255 ± 0.109
   Curly-FM                13.542 ± 0.229  7.407 ± 0.274
   FFM (ours)              12.785 ± 0.088  8.034 ± 0.367
   FFM (ours), with noise  12.832 ± 0.122  7.725 ± 0.167
   sampling floor                   6.753          7.028

## Intermediate-state interpolation (Table 1, Pancreas)

Top: midpoint $W_2$ on Pancreas; bottom: matched-information objective ablation on Pancreas at $d = 20$. Mean ± population SD over five seeds. The ablation is a separate run set at the matched-ablation operating point (Table 8), so its FFM row differs from the top.

In [4]:
paper.table1()

Pancreas
d  Method                                
20 OT-CFM                  12.335 ± 0.052
   MFM / LAND              11.908 ± 0.297
   Curly-FM                12.370 ± 0.076
   FFM (ours)              10.978 ± 0.166
   FFM (ours), with noise  10.942 ± 0.075
50 OT-CFM                  13.496 ± 0.107
   MFM / LAND              13.104 ± 0.127
   Curly-FM                13.542 ± 0.229
   FFM (ours)              12.785 ± 0.088
   FFM (ours), with noise  12.832 ± 0.122

In [5]:
paper.table1_bottom()

,Shared coupling,Method-specific coupling
Method,,
Graph-drift FM (control),11.92 ± 0.06,11.98 ± 0.06
Anisotropic fixed-time FM,11.51 ± 0.09,11.51 ± 0.07
Second-moment FFM (m = 0),11.65 ± 0.09,11.68 ± 0.12
"FFM, one-form deleted",11.29 ± 0.10,11.24 ± 0.10
FFM (ours),10.94 ± 0.15,10.93 ± 0.14


## Fate prediction (Table 2, Pancreas)

Source and target are the middle and last pseudotime tertiles (the first is not used), with the RNA velocity and transition graph recomputed on these cells. For each source cell, [SF]$^2$M and FFM with noise sample 64 trajectories at noise width 0.3 at $d = 20$, chosen from a width sweep evaluated on the scored cells, and at the validation-tuned 0.03 at $d = 50$; deterministic FFM integrates one; fate probabilities come from the labels of nearby target cells. Centered MSE (cMSE) against the CellRank reference, which removes differences in average fate composition, is computed on the later half of the source cells by latent time (616 of 1,232), because the earlier half has an almost constant reference; Constant assigns every cell the mean reference fate vector of the source cells. Mean ± population SD over five seeds.

In [6]:
paper.table2()

,d = 20,d = 50
Method,,
Constant,0.160,0.160
[SF]2M,0.129 ± 0.020,0.327 ± 0.004
FFM,0.519 ± 0.030,0.541 ± 0.018
"FFM + velocity, with noise",0.098 ± 0.009,0.366 ± 0.019


## Wall-clock time (Table 4)

Wall-clock time in seconds per seed on an NVIDIA A100, mean over five seeds. The stochastic FFM arm reuses the deterministic interpolant and coupling.

In [7]:
paper.table4()

,d = 2,d = 10,d = 20,d = 50
Method,,,,
OT-CFM,3,3,3,3
MFM / LAND,21,62,113,266
Curly-FM,38,108,193,461
"FFM (ours), deterministic",25,64,112,262
"FFM (ours), with noise",13,29,49,110


## Robustness analyses (Figure 4)

(a) Intermediate-state reconstruction at representative model times in $[1/3, 2/3]$, from the five-seed runs of the midpoint table; the final column averages $W_2$ over all 11 equally spaced evaluation times. (b) Sensitivity to the conditional-path noise width at $d = 20$ for anisotropic and isotropic path covariance, and (c) ablation of the conditional-path and inference covariance at noise width 0.1. Panels (b) and (c) are a separate three-seed run set with their own deterministic interpolants, scored against the same 1,112 withheld cells.

In [8]:
paper.figure4a()

W2(1/3)         W2(0.433)         W2(0.567)  \
d  Method                                                                     
2  FFM, deterministic   1.7753 ± 0.1603   1.6879 ± 0.1374   1.6764 ± 0.1736   
   FFM, with noise      1.8705 ± 0.1865   1.7464 ± 0.2069   1.7765 ± 0.0775   
   OT-CFM               1.8393 ± 0.0228   1.7846 ± 0.0222   1.7522 ± 0.0208   
   MFM / LAND           1.5371 ± 0.0176   1.5030 ± 0.0168   1.4579 ± 0.0177   
   Curly-FM             1.7990 ± 0.0374   1.7239 ± 0.0411   1.6745 ± 0.0440   
20 FFM, deterministic  12.0615 ± 0.1811  11.3252 ± 0.1769  10.8351 ± 0.1567   
   FFM, with noise     12.0468 ± 0.1174  11.2974 ± 0.0876  10.8001 ± 0.1048   
   OT-CFM              13.0826 ± 0.1043  12.5523 ± 0.0764  12.2528 ± 0.0281   
   MFM / LAND          12.3544 ± 0.3515  12.0502 ± 0.3233  11.8044 ± 0.2733   
   Curly-FM            13.0473 ± 0.0549  12.5575 ± 0.0605  12.3118 ± 0.0971   
50 FFM, deterministic  13.5017 ± 0.1195  12.9802 ± 0.0978  12.7484 ± 0.0942   
   FFM, with noise     13.5085 ± 0.1416  13.0120 ± 0.1281  12.8085 ± 0.1244   
   OT-CFM              14.1231 ± 0.1192  13.6682 ± 0.1112  13.4509 ± 0.1054   
   MFM / LAND          13.9104 ± 0.1204  13.3588 ± 0.1234  12.9660 ± 0.1331   
   Curly-FM            14.0924 ± 0.0827  13.6825 ± 0.1638  13.5140 ± 0.2891   

                                W2(2/3)       W2[1/3,2/3]  
d  Method                                                  
2  FFM, deterministic   1.6996 ± 0.1358   1.6991 ± 0.1129  
   FFM, with noise      1.7920 ± 0.0581   1.7804 ± 0.1037  
   OT-CFM               1.7621 ± 0.0194   1.7775 ± 0.0213  
   MFM / LAND           1.4143 ± 0.0220   1.4791 ± 0.0168  
   Curly-FM             1.6844 ± 0.0447   1.7113 ± 0.0420  
20 FFM, deterministic  11.1550 ± 0.2103  11.2195 ± 0.1615  
   FFM, with noise     11.1539 ± 0.2360  11.1943 ± 0.0769  
   OT-CFM              12.3773 ± 0.0415  12.4952 ± 0.0469  
   MFM / LAND          11.7134 ± 0.2474  11.9572 ± 0.2950  
   Curly-FM            12.4453 ± 0.1314  12.5229 ± 0.0737  
50 FFM, deterministic  12.9978 ± 0.1384  12.9655 ± 0.0929  
   FFM, with noise     13.0620 ± 0.1536  13.0084 ± 0.1253  
   OT-CFM              13.6124 ± 0.1099  13.6468 ± 0.1051  
   MFM / LAND          12.9883 ± 0.1525  13.2435 ± 0.1262  
   Curly-FM            13.6544 ± 0.3472  13.6760 ± 0.2213

In [9]:
paper.figure4b()

M_t = C_ρ(μ_t)                               M_t = I  \
                  W2(1/2)              W2(1)            W2(1/2)   
Width                                                             
0.003    10.9974 ± 0.1018    5.2929 ± 0.2290   11.0159 ± 0.0776   
0.010    11.0100 ± 0.0715    5.3224 ± 0.3146   11.0813 ± 0.1492   
0.030    11.0227 ± 0.1909    5.3127 ± 0.2379   10.9665 ± 0.1032   
0.100    10.9619 ± 0.0989    5.0594 ± 0.2667   10.9324 ± 0.1241   
0.300    11.0192 ± 0.1705    5.4535 ± 0.2132   10.9866 ± 0.1762   
1.000    14.3821 ± 0.0429    7.4957 ± 0.1703   14.3339 ± 0.0859   
3.000    34.7424 ± 0.4379   23.6222 ± 0.7989   34.4706 ± 0.5384   
10.000  129.1656 ± 2.1831  106.9728 ± 4.7193  144.6216 ± 3.1283   

                           
                    W2(1)  
Width                      
0.003     5.2240 ± 0.0412  
0.010     5.2182 ± 0.2159  
0.030     5.3847 ± 0.0722  
0.100     4.8931 ± 0.0907  
0.300     5.1528 ± 0.0638  
1.000     7.3343 ± 0.1336  
3.000    26.6268 ± 0.5555  
10.000  131.2148 ± 4.3699

In [10]:
paper.figure4c()

W2(1/2)            W2(1)
Path covariance Inference covariance                                   
C_ρ(μ_t)        C_ρ(X_t)              10.9619 ± 0.0989  5.0594 ± 0.2667
                I                     10.9427 ± 0.1123  4.9708 ± 0.1974
I               C_ρ(X_t)              10.9481 ± 0.1155  4.9059 ± 0.1588
                I                     10.9324 ± 0.1241  4.8931 ± 0.0907

## Sensitivity to $\rho$ and $\lambda$ (Table 5)

One-at-a-time sensitivity of FFM on Pancreas; the reported value is the validation selection objective, lower is better. Values are multiples of $\rho^*$, the mean eigenvalue of the second moment $C$, and of $\bar a$, the mean first-moment norm $\lVert m \rVert_{C_\rho^{-1}}$; the selected value is marked with an asterisk.

In [11]:
paper.table5()

Fixed parameter                               Values  \
d  Varied parameter                                                        
2  ρ                      λ = 0.001            0.1, 0.3, 1.0*, 3.0, 10.0   
   λ                        ρ = 1.0  0.0001, 0.0003, 0.001*, 0.003, 0.01   
20 ρ                        λ = 1.0           0.01, 0.03, 0.1*, 0.3, 1.0   
   λ                        ρ = 0.1            0.03, 0.1, 0.3, 1.0*, 3.0   
50 ρ                        λ = 0.3       0.001, 0.003, 0.01*, 0.03, 0.1   
   λ                       ρ = 0.01            0.03, 0.1, 0.3*, 1.0, 3.0   

                                        Validation objective  
d  Varied parameter                                           
2  ρ                      0.886, 1.021, 0.811*, 1.018, 1.065  
   λ                      1.104, 1.004, 0.811*, 0.937, 0.902  
20 ρ                      8.396, 8.376, 8.330*, 8.413, 8.533  
   λ                      8.824, 8.774, 8.836, 8.330*, 8.800  
50 ρ                 10.449, 10.439, 10.413*, 10.437, 10.753  
   λ                 10.546, 10.513, 10.413*, 10.557, 10.640

## Intermediate-state support (Table 6)

Midpoint Euclidean $W_2$ when 0%, 5%, or 35% of the intermediate population is available during training. This is a separate three-seed run set scored against all remaining withheld cells (1,232, 1,172 and 802), including those of the validation subset, so its 0% column differs from the midpoint table and Table 1.

In [12]:
paper.table6()

0%            5%           35%
d  Method                                                          
2  OT-CFM                   1.76 ± 0.02   1.76 ± 0.03   1.77 ± 0.02
   MFM / LAND               1.47 ± 0.01   1.45 ± 0.05   0.69 ± 0.03
   Curly-FM                 1.68 ± 0.03   1.67 ± 0.04   1.65 ± 0.04
   FFM (ours)               1.67 ± 0.10   1.42 ± 0.09   1.38 ± 0.06
   FFM (ours), with noise   1.85 ± 0.10   1.51 ± 0.29   1.37 ± 0.20
   Sampling floor                  0.11          0.11          0.11
20 OT-CFM                  12.33 ± 0.06  12.33 ± 0.04  12.35 ± 0.07
   MFM / LAND              11.86 ± 0.31  12.13 ± 0.08  11.38 ± 0.19
   Curly-FM                12.39 ± 0.10  12.27 ± 0.15  13.63 ± 0.04
   FFM (ours)              10.96 ± 0.14   9.92 ± 0.22   9.27 ± 0.26
   FFM (ours), with noise  10.90 ± 0.06   9.64 ± 0.23   9.01 ± 0.19
   Sampling floor                  4.12          4.16          4.14
50 OT-CFM                  13.53 ± 0.04  13.53 ± 0.05  13.56 ± 0.09
   MFM / LAND              13.12 ± 0.08  13.27 ± 0.08  13.13 ± 0.04
   Curly-FM                13.60 ± 0.27  13.35 ± 0.16  15.39 ± 0.10
   FFM (ours)              12.78 ± 0.06  12.44 ± 0.08  11.60 ± 0.07
   FFM (ours), with noise  12.76 ± 0.08  12.33 ± 0.09  11.51 ± 0.06
   Sampling floor                  6.72          6.74          6.70

## Matched-information objective ablation (Table 8)

Midpoint $W_2$ against the 1,112 scored withheld cells with a shared coupling and with a method-specific coupling, each objective at the matched-ablation operating point selected for it under that coupling. The points at $d = 2$ and $50$ were selected before the velocity refit and reused. Mean ± population SD over ten seeds at $d = 2$ and five seeds at $d = 20$ and $50$.

In [13]:
paper.table8()

d = 2                     d = 20  \
                                Shared     Specific        Shared   
Objective                                                           
Graph-drift FM             1.76 ± 0.08  1.90 ± 0.08  11.92 ± 0.06   
Anisotropic fixed-time FM  1.73 ± 0.05  1.73 ± 0.04  11.51 ± 0.09   
Second-moment FFM (m = 0)  1.73 ± 0.07  1.59 ± 0.07  11.65 ± 0.09   
FFM, one-form deleted      1.88 ± 0.12  1.86 ± 0.13  11.29 ± 0.10   
FFM (ours)                 1.68 ± 0.12  1.70 ± 0.16  10.94 ± 0.15   

                                               d = 50                
                               Specific        Shared      Specific  
Objective                                                            
Graph-drift FM             11.98 ± 0.06  13.44 ± 0.08  13.51 ± 0.08  
Anisotropic fixed-time FM  11.51 ± 0.07  12.78 ± 0.10  12.76 ± 0.10  
Second-moment FFM (m = 0)  11.68 ± 0.12  12.77 ± 0.11  12.83 ± 0.10  
FFM, one-form deleted      11.24 ± 0.10  12.83 ± 0.07  12.77 ± 0.07  
FFM (ours)                 10.93 ± 0.14  12.89 ± 0.11  12.84 ± 0.07

## WLF and GSBM (Table 10, Pancreas)

The Finsler geometry inside Wasserstein Lagrangian Flows and Generalized Schrödinger Bridge Matching, against their Euclidean versions, at $d = 20$; five seeds. FFM and the Finsler variants run at the matched-ablation operating point of FFM with the method-specific coupling ($\rho = 0.01\,\rho^*$, $\lambda = \bar a$), so the FFM row differs from Table 1.

In [14]:
paper.table10()

,Mid W2,End W2,Time (s)
Method,,,
FFM (ours),10.93 ± 0.14,5.31 ± 0.26,110
Finsler-WLF,10.94 ± 0.17,5.43 ± 0.34,69
Finsler-GSBM,11.03 ± 0.19,6.40 ± 0.47,95
Euclidean WLF,12.45 ± 0.05,5.30 ± 0.20,62
Euclidean GSBM,12.34 ± 0.10,5.45 ± 0.31,16
